---

# **[실습] 주택청약 FAQ 시스템 구현**

### **문제 설명**
이전 코드를 기반으로 주택청약 FAQ 시스템을 다음 요구사항에 맞춰 개선합니다. 

1. 응답 품질 향상 (1개 이상)
   - 생성된 답변의 품질을 평가 (답변이 불충분한 경우 예외 처리)
   - 관련성 높은 FAQ 문서 검색 (임베딩 모델, 청크 크기, 벡터 검색 방법 등)

2. 사용자 경험 개선 (1개 이상)
   - 대화 이력 관리 기능 추가 (요약, 트리밍 기능 등 고려)
   - 최근 대화 기반 컨텍스트 구성 
   - 사용자 프로필 기반 맞춤 응답

### **제약 조건**
- Gradio ChatInterface 사용
- RAG 구조 유지

In [ ]:
# 표준 라이브러리
import re
from glob import glob
from dataclasses import dataclass
from typing import Dict, List

# 환경변수 관리
from dotenv import load_dotenv

# 웹 인터페이스
import gradio as gr

# 데이터 모델링
from pydantic import BaseModel, Field

# LLM 모델
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 문서 처리
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

# 프롬프트 및 출력 처리
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import tiktoken

# 벡터 저장소
from langchain_chroma import Chroma

# 환경변수 로드
load_dotenv()

# LLM 초기화
llm = ChatOpenAI(
    model='gpt-4.1-mini',
    temperature=0.1,
    top_p=0.9, 
)

# 데이터 파일 로드
data_files = glob("../data/housing_faq.txt")
loader = TextLoader(data_files[0], 'utf-8')
docs = loader.load()

# 문서 전처리 함수
def extract_qa_pairs(text):
    qa_pairs = []
    
    # 텍스트를 라인별로 분리하고 각 라인의 앞뒤 공백 제거
    lines = [line.strip() for line in text.split('\n')]
    current_question = None
    current_answer = []
    current_number = None
    in_answer = False
    
    for i, line in enumerate(lines):
        if not line:  # 빈 라인 처리
            if in_answer and current_answer and i + 1 < len(lines) and lines[i + 1].startswith('Q'):
                # 다음 질문이 시작되기 전 빈 줄이면 현재 QA 쌍 저장
                qa_pairs.append({
                    'number': current_number,
                    'question': current_question,
                    'answer': ' '.join(current_answer).strip()
                })
                in_answer = False
                current_answer = []
            continue
            
        # 새로운 질문 확인 (Q 다음에 숫자가 오는 패턴)
        q_match = re.match(r'Q(\d+)\s+(.*)', line)
        if q_match:
            # 이전 QA 쌍이 있으면 저장
            if current_question is not None and current_answer:
                qa_pairs.append({
                    'number': current_number,
                    'question': current_question,
                    'answer': ' '.join(current_answer).strip()
                })
            
            # 새로운 질문 시작
            current_number = int(q_match.group(1))
            current_question = q_match.group(2).strip().rstrip('?') + '?'  # 질문 마크 정규화
            current_answer = []
            in_answer = False
            
        # 답변 시작 확인
        elif line.startswith('A ') or (current_question and not current_answer and line):
            in_answer = True
            current_answer.append(line.lstrip('A '))
            
        # 기존 답변에 내용 추가
        elif current_question is not None and (in_answer or not line.startswith('Q')):
            if in_answer or (current_answer and not line.startswith('Q')):
                current_answer.append(line)
    
    # 마지막 QA 쌍 처리
    if current_question is not None and current_answer:
        qa_pairs.append({
            'number': current_number,
            'question': current_question,
            'answer': ' '.join(current_answer).strip()
        })
    
    # 번호 순서대로 정렬
    qa_pairs.sort(key=lambda x: x['number'])
    
    return qa_pairs

# 응답 품질 평가 모델
class ResponseQuality(BaseModel):
    relevance_score: float = Field(description="답변의 관련성 점수 (0-1)")
    completeness_score: float = Field(description="답변의 완성도 점수 (0-1)")
    confidence_score: float = Field(description="답변의 신뢰도 점수 (0-1)")
    overall_quality: str = Field(description="전체 품질 평가 (excellent/good/fair/poor)")
    improvement_suggestions: str = Field(description="개선 제안사항")

# 사용자 프로필 모델
@dataclass
class UserProfile:
    interests: List[str]
    experience_level: str
    recent_topics: List[str]
    preferred_detail_level: str

# 대화 이력 관리
class ConversationManager:
    def __init__(self, max_history: int = 10, max_tokens: int = 2000):
        self.max_history = max_history
        self.max_tokens = max_tokens
        self.conversation_history: List[Dict[str, str]] = []
        self.user_profiles: Dict[str, UserProfile] = {}
        self.summarized_history: Dict[str, str] = {}  # 사용자별 요약된 이력
        
        # 토큰 계산기
        try:
            self.encoding = tiktoken.encoding_for_model("gpt-4o-mini")
        except:
            self.encoding = tiktoken.get_encoding("cl100k_base")

    def count_tokens(self, text: str) -> int:
        """텍스트의 토큰 수 계산"""
        return len(self.encoding.encode(text))
    
    def summarize_old_conversations(self, user_id: str, conversations_to_summarize: List[Dict]) -> str:
        """오래된 대화들을 요약"""
        if not conversations_to_summarize:
            return ""
        
        summary_text = ""
        for conv in conversations_to_summarize:
            summary_text += f"Q: {conv['question']}\nA: {conv['answer'][:100]}...\n\n"
        
        # LLM을 사용한 요약
        summary_prompt = ChatPromptTemplate.from_messages([
            ("system", "다음 주택청약 관련 대화들을 핵심 내용만 간단히 요약해주세요. 사용자가 관심있어 했던 주제와 주요 질문들을 중심으로 3-4줄로 정리하세요."),
            ("user", f"요약할 대화:\n{summary_text}")
        ])
        
        try:
            summary_chain = summary_prompt | llm | StrOutputParser()
            summary = summary_chain.invoke({})
            return summary
        except:
            return "이전 대화에서 청약 관련 다양한 질문들이 있었습니다."
    
    def add_exchange(self, user_id: str, question: str, answer: str, topics: List[str] = None):
        exchange = {
            "user_id": user_id,
            "question": question,
            "answer": answer,
            "topics": topics or [],
            "timestamp": "now"
        }
        self.conversation_history.append(exchange)
        
        # 토큰 기반 관리
        self.manage_history_by_tokens(user_id)
        
        # 기존 프로필 업데이트
        self.update_user_profile(user_id, topics or [])

    def manage_history_by_tokens(self, user_id: str):
        """토큰 수 기반으로 이력 관리"""
        user_conversations = [h for h in self.conversation_history if h["user_id"] == user_id]
        
        # 현재 대화들의 총 토큰 수 계산
        total_tokens = 0
        for conv in user_conversations:
            total_tokens += self.count_tokens(conv['question'] + conv['answer'])
        
        # 토큰 수가 초과하면 처리
        if total_tokens > self.max_tokens or len(user_conversations) > self.max_history:
            # 오래된 대화 절반을 요약
            conversations_to_summarize = user_conversations[:len(user_conversations)//2]
            
            if conversations_to_summarize:
                # 요약 생성
                new_summary = self.summarize_old_conversations(user_id, conversations_to_summarize)
                
                # 기존 요약과 합치기
                if user_id in self.summarized_history:
                    self.summarized_history[user_id] = f"{self.summarized_history[user_id]}\n\n{new_summary}"
                else:
                    self.summarized_history[user_id] = new_summary
                
                # 요약된 대화들을 실제 이력에서 제거
                for conv in conversations_to_summarize:
                    if conv in self.conversation_history:
                        self.conversation_history.remove(conv)
    
    def update_user_profile(self, user_id: str, topics: List[str]):
        if user_id not in self.user_profiles:
            self.user_profiles[user_id] = UserProfile(
                interests=[],
                experience_level="beginner",
                recent_topics=[],
                preferred_detail_level="detailed"
            )
        
        profile = self.user_profiles[user_id]
        profile.recent_topics.extend(topics)
        profile.recent_topics = profile.recent_topics[-5:]  # 최근 5개만 유지
    
    def get_context(self, user_id: str) -> str:
        context_parts = []
        
        # 요약된 이전 대화
        if user_id in self.summarized_history and self.summarized_history[user_id]:
            context_parts.append(f"이전 대화 요약:\n{self.summarized_history[user_id]}")
        
        # 최근 상세 대화 (최근 5개)
        recent_history = [h for h in self.conversation_history[-3:] if h["user_id"] == user_id]
        if recent_history:
            recent_context = "최근 대화:\n"
            for i, h in enumerate(recent_history, 1):
                recent_context += f"{i}. Q: {h['question']}\n   A: {h['answer'][:100]}...\n\n"
            context_parts.append(recent_context)
        
        return "\n\n---\n\n".join(context_parts)

    def get_user_profile(self, user_id: str) -> UserProfile:
        """사용자 프로필 조회"""
        if user_id not in self.user_profiles:
            self.user_profiles[user_id] = UserProfile(
                interests=["주택청약", "분양"],
                experience_level="beginner",
                recent_topics=[],
                preferred_detail_level="detailed"
            )
        return self.user_profiles[user_id]

# QA 쌍 추출
qa_pairs = extract_qa_pairs(docs[0].page_content) 

# Document 객체로 변환
formatted_docs = []
for qa in qa_pairs:
    doc_content = f"질문: {qa['question']}\n답변: {qa['answer']}"
    doc = Document(
        page_content=doc_content,
        metadata={
            "question": qa['question'],
            "answer": qa['answer'],
            "number": qa['number']
        }
    )
    formatted_docs.append(doc)

# 임베딩 모델
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 벡터 저장소 로드
vector_store = Chroma.from_documents(  
    documents=formatted_docs,
    embedding=embeddings,
    collection_name="housing_faq_db",
    persist_directory="./chroma_db",
)

# 검색기 생성
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# 응답 품질 평가 체인
quality_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 답변 품질을 평가하는 전문가입니다.
주어진 질문과 답변을 분석하여 다음 기준으로 평가해주세요:
- 관련성: 답변이 질문과 얼마나 관련이 있는가
- 완성도: 답변이 얼마나 완전한가
- 신뢰도: 답변이 얼마나 신뢰할 만한가"""),
    ("user", "질문: {question}\n답변: {answer}\n\n위 답변을 평가해주세요.")
])

quality_evaluator = quality_prompt | llm.with_structured_output(ResponseQuality)

# RAG 체인
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 주택청약 전문 상담사입니다.
제공된 FAQ 정보를 바탕으로 사용자의 질문에 정확하고 도움이 되는 답변을 제공하세요.
     
**중요**: 질문이 주택청약과 관련이 없다면 정중히 안내하고 주택청약 관련 질문을 요청하세요.

주택청약 관련 질문(청약통장, 분양, 당첨 조건, 전매 등)에만 답변하고,
그 외 질문에는 "주택청약 관련 질문에만 답변드릴 수 있습니다"라고 응답하세요.

**사용자 프로필:**
- 경험 수준: {experience_level}
- 관심 분야: {interests}
- 선호 설명 수준: {detail_level}

**답변 가이드라인:**
- 사용자의 경험 수준에 맞춰 설명하세요
- 정확한 정보를 제공하되, 이해하기 쉽게 설명하세요
- 관련 법령이나 정책이 변경될 수 있음을 언급하세요
- 구체적인 사례나 예시를 포함하여 설명하세요
- 추가 문의가 필요한 경우 관련 기관을 안내하세요

**컨텍스트 정보:**
{context}

**사용자 대화 이력:**
{user_context}"""),
    
    ("user", "{question}")
])

conversation_manager = ConversationManager()

# 답변 생성 함수
def answer_question(question: str, user_id: str = "default_user") -> str:
    try:
        # 관련 문서 검색
        relevant_docs = retriever.get_relevant_documents(question)
        
        # 컨텍스트 구성
        context = "\n\n".join([doc.page_content for doc in relevant_docs])
        user_context = conversation_manager.get_context(user_id)
        
        # 사용자 프로필 정보 가져오기
        user_profile = conversation_manager.get_user_profile(user_id)
        
        # 답변 생성
        rag_chain = rag_prompt | llm | StrOutputParser()
        answer = rag_chain.invoke({
            "question": question,
            "context": context,
            "user_context": user_context,
            "experience_level": user_profile.experience_level,
            "interests": ", ".join(user_profile.interests) if user_profile.interests else "주택청약 일반",
            "detail_level": user_profile.preferred_detail_level
        })
        
        # 품질 평가
        try:
            quality_result = quality_evaluator.invoke({
                "question": question,
                "answer": answer
            })
            
            # 답변이 너무 부족한 경우 경고 추가
            if quality_result.get("relevance_score", 0.8) < 0.4:
                answer += "\n\n💡 더 구체적인 질문을 해주시면 더 정확한 답변을 드릴 수 있습니다."
                
        except Exception as e:
            print(f"품질 평가 오류: {e}")
        
        # 대화 이력에 추가
        conversation_manager.add_exchange(user_id, question, answer)
        
        return answer
        
    except Exception as e:
        return f"죄송합니다. 답변 생성 중 오류가 발생했습니다: {str(e)}"


def chat_interface(message, history):
    "Gradio 채팅 인터페이스"
    user_id = "default_user"
    
    # History 동기화 함수 호출
    sync_gradio_history_to_manager(history, user_id)

    # answer_question 함수 호출
    response = answer_question(message, user_id)
    
    return response

# History 동기화 함수
def sync_gradio_history_to_manager(gradio_history, user_id: str):
    
    # 현재 ConversationManager의 이력 길이
    current_manager_length = len([h for h in conversation_manager.conversation_history if h["user_id"] == user_id])
    
    # Gradio history가 더 길면 동기화 필요
    if len(gradio_history) > current_manager_length:
        # 새로운 대화만 추가
        for i in range(current_manager_length, len(gradio_history)):
            if i < len(gradio_history):
                user_msg, assistant_msg = gradio_history[i]
                if user_msg and assistant_msg:  # 둘 다 존재할 때만 추가
                    conversation_manager.add_exchange(
                        user_id=user_id,
                        question=user_msg,
                        answer=assistant_msg,
                        topics=[]
                    )

def create_gradio_interface():
    """Gradio 인터페이스 생성"""
    with gr.Blocks(title="주택청약 FAQ 챗봇") as demo:
        gr.Markdown("""
        # 🏠 주택청약 FAQ 챗봇
        
        주택청약에 관한 궁금한 점을 물어보세요!
        - 청약통장, 청약 조건, 분양 정보 등
        - 대화 이력을 바탕으로 맞춤형 답변 제공
        - 답변 품질을 실시간으로 평가
        - 사용자 프로필 기반 개인화 답변
        """)
        
        # 채팅 인터페이스
        chatbot = gr.ChatInterface(
            fn=chat_interface,
            title="",
            description="",
            examples=[
                "청약통장이란 무엇인가요?",
                "1순위 청약 조건을 알려주세요",
                "분양권 전매는 언제부터 가능한가요?",
                "청약 당첨 확률을 높이는 방법이 있나요?"
            ],
            submit_btn="질문하기",
        )
        
        # 시스템 정보
        with gr.Accordion("시스템 정보", open=False):
            gr.Markdown(f"""
            - **벡터 DB**: {len(formatted_docs)}개 FAQ 문서 저장됨
            - **검색 방식**: 유사도 기반 상위 3개 문서 검색
            - **대화 이력**: 최근 10개 대화 저장
            - **품질 평가**: 관련성, 완성도, 신뢰도 자동 평가
            - **개인화**: 사용자 프로필 기반 맞춤형 답변
            """)
    
    return demo

# 메인 실행
if __name__ == "__main__":    
    # Gradio 앱 실행
    demo = create_gradio_interface()
    demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=False,
        debug=True
    )